<a href="https://colab.research.google.com/github/PepBiel/NLP_project/blob/main/Fornes_Reynes_JosepGabriel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Título**: Ajuste fino a T5 para Resumenes

**Alumno**: Josep Gabriel Fornes Reynes

# Introducción

In [1]:
!pip install -q transformers datasets accelerate evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.7 MB/s eta 0:00:00


In [2]:
# Importar las clases de Hugging Face
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [3]:
# Nombre del modelo en el Hub de Hugging Face
model_checkpoint = "t5-small"

In [4]:
# 1. Cargar el Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

In [5]:
# 2. Cargar el Modelo (aquí se descargan los pesos)
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

# Dataset

In [6]:
def preprocess_function(examples):
    inputs = ["summarize: " + doc for doc in examples["article"]] # 'article' depende del nombre de la columna en tu dataset
    model_inputs = tokenizer(inputs, max_length=1024, truncation=True)

    # Tokenizar los objetivos (los resúmenes reales)
    labels = tokenizer(text_target=examples["highlights"], max_length=128, truncation=True) # 'highlights' es la columna del resumen

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [7]:
from transformers import DataCollatorForSeq2Seq
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

In [8]:
from datasets import load_dataset

# Cargar el dataset desde Hugging Face Hub
# "3.0.0" es la versión específica que suele usarse
dataset = load_dataset("cnn_dailymail", "3.0.0")

# Ver la estructura de un ejemplo
print(dataset["train"][0])
print(dataset["test"][1])

README.md: 0.00B [00:00, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

3.0.0/validation-00000-of-00001.parquet:   0%|          | 0.00/34.7M [00:00<?, ?B/s]

3.0.0/test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

{'article': 'LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won\'t cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fast cars, drink and celebrity parties. "I don\'t plan to be one of those people who, as soon as they turn 18, suddenly buy themselves a massive sports car collection or something similar," he told an Australian interviewer earlier this month. "I don\'t think I\'ll be particularly extravagant. "The things I like buying are things that cost about 10 pounds -- books and CDs and DVDs." At 18, Radcliffe will be able to gamble in a casino, buy a drink in a pub or see the horror film "Hostel: Part II," currently six places below his number one movie on the UK box office char

In [9]:
# Tomar solo 1000 ejemplos para entrenamiento y 200 para validación
# Esto acelera drásticamente el proceso en Colab
small_train_dataset = dataset["train"].shuffle(seed=42).select(range(1000))
small_eval_dataset = dataset["validation"].shuffle(seed=42).select(range(200))

print(f"Ejemplos de entrenamiento: {len(small_train_dataset)}")
print(f"Ejemplos de validación: {len(small_eval_dataset)}")

Ejemplos de entrenamiento: 1000
Ejemplos de validación: 200


In [10]:
# Aplicar la tokenización a todo el dataset (usando el subconjunto que creaste antes)
tokenized_train = small_train_dataset.map(preprocess_function, batched=True)
tokenized_eval = small_eval_dataset.map(preprocess_function, batched=True)

# Verificar que funcionó
print(tokenized_train[0].keys())

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

dict_keys(['article', 'highlights', 'id', 'input_ids', 'attention_mask', 'labels'])


# Implementación

In [11]:
!pip install rouge_score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=f041d72c9b42afd2823b03f8e0bd2fa8ea85b7c03384fc993613774602c1dd89
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [12]:
import evaluate
import numpy as np

# Cargar la métrica ROUGE
rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    # Decodificar las predicciones (de números a texto)
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)

    # Reemplazar -100 en las etiquetas porque el tokenizer no puede decodificarlos
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Calcular ROUGE
    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)

    # Formatear un poco los números para que se vean bien
    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in predictions]
    result["gen_len"] = np.mean(prediction_lens)

    return {k: round(v, 4) for k, v in result.items()}

In [13]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

training_args = Seq2SeqTrainingArguments(
    output_dir="modelo_resumen_t5",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=2,
    predict_with_generate=True,
    fp16=True,
    report_to="none"
)

In [14]:
# Coger un artículo del conjunto de prueba
idx = 0
input_text = small_eval_dataset[idx]["article"]
target_text = small_eval_dataset[idx]["highlights"]

# Preparar el texto para el modelo
input_ids = tokenizer("summarize: " + input_text, return_tensors="pt").input_ids.to("cuda")

# Generar el resumen
outputs = model.generate(input_ids, max_length=128)
generated_summary = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("--- ARTÍCULO ORIGINAL (Fragmento) ---")
print(input_text[:500] + "...") # Solo imprimimos el principio
print("\n--- RESUMEN HUMANO (Real) ---")
print(target_text)
print("\n--- RESUMEN GENERADO POR TU MODELO ---")
print(generated_summary)

RuntimeError: Expected all tensors to be on the same device, but got index is on cuda:0, different from other tensors on cpu (when checking argument in method wrapper_CUDA__index_select)

# Resultados y discusión

# Conclusiones